# Notebook 02  Autoencoder Pre-training
**Phase 1:** Train CNN Encoder + Decoder on Normal clips only (MSE reconstruction loss).

## 0. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.config import (
    MODELS_DIR, PLOTS_DIR, RESULTS_DIR, SEED, CLIP_LEN, CLASS_TO_IDX
)
from src.model import Encoder, Decoder, Autoencoder
from src.dataset import load_normal_npy, load_npy_split
from src.train import pretrain_autoencoder, plot_loss
from src.evaluate import (
    compute_reconstruction_errors, compute_threshold,
    plot_reconstruction_samples, plot_mse_distribution
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"PyTorch: {torch.__version__}")
print(f"Device : {DEVICE}")

PyTorch: 2.13.0+cu130
Device : cuda


## 1. Load Data

In [2]:
X_normal = load_normal_npy()
X_val, y_val = load_npy_split("val")
X_val_normal = X_val[y_val == CLASS_TO_IDX["Normal"]]
X_val_anom   = X_val[y_val != CLASS_TO_IDX["Normal"]]

print(f"Normal train clips : {X_normal.shape}")
print(f"Val Normal clips   : {X_val_normal.shape}")
print(f"Val Anomaly clips  : {X_val_anom.shape}")

Normal train clips : (3000, 8, 64, 64, 3)
Val Normal clips   : (450, 8, 64, 64, 3)
Val Anomaly clips  : (1800, 8, 64, 64, 3)


## 2. Build Autoencoder

In [3]:
encoder     = Encoder()
decoder     = Decoder()
autoencoder = Autoencoder(encoder, decoder)

total = sum(p.numel() for p in autoencoder.parameters())
print(f"Total parameters: {total:,}")
print(autoencoder)

Total parameters: 4,461,475
Autoencoder(
  (encoder): Encoder(
    (cnn): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): ReLU()
      (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (fc): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=8192, out_features=256, bias=True)
      (2): ReLU()
    )
  )
  (decoder): Decoder(
    (fc): Sequential(
      (0): Linear(in_features=256, out_features=8192, bias=True)
      (1): ReLU()
    )
    (deconv): Sequential(
      (0): ConvTranspose2d(128, 64, kernel_size

## 3. Pre-train

In [4]:
history = pretrain_autoencoder(autoencoder, X_normal, val_split=0.15, device=DEVICE)

Epoch 01/30  loss=0.04378  val_loss=0.02911
Epoch 05/30  loss=0.00910  val_loss=0.01601
Epoch 10/30  loss=0.00639  val_loss=0.01423
Epoch 15/30  loss=0.00530  val_loss=0.01365
Epoch 20/30  loss=0.00466  val_loss=0.01314
Epoch 25/30  loss=0.00424  val_loss=0.01255
Epoch 30/30  loss=0.00393  val_loss=0.01253


## 4. Loss Curves

In [5]:
plot_loss(history, title="Autoencoder Pre-training Loss",
          save_path=PLOTS_DIR / "pretrain_loss.png")
plt.show()

[OK] Plot saved: /home/mjl/softwarica/ANN/outputs/plots/pretrain_loss.png


## 5. Visual Quality Check

In [6]:
# Reload best weights
autoencoder.load_state_dict(
    torch.load(MODELS_DIR / "autoencoder_best.pth", map_location=DEVICE))
autoencoder = autoencoder.to(DEVICE)

plot_reconstruction_samples(autoencoder, X_val_normal, n=4,
    save_path=PLOTS_DIR / "reconstructions_normal.png", device=DEVICE)
plt.show()

plot_reconstruction_samples(autoencoder, X_val_anom[:20], n=4,
    save_path=PLOTS_DIR / "reconstructions_anomaly.png", device=DEVICE)
plt.show()

[OK] Reconstructions saved: /home/mjl/softwarica/ANN/outputs/plots/reconstructions_normal.png
[OK] Reconstructions saved: /home/mjl/softwarica/ANN/outputs/plots/reconstructions_anomaly.png


## 6. MSE Distribution: Normal vs Anomalous

In [7]:
plot_mse_distribution(autoencoder, X_val_normal, X_val_anom,
    save_path=PLOTS_DIR / "mse_distribution.png", device=DEVICE)
plt.show()

[OK] MSE distribution saved: /home/mjl/softwarica/ANN/outputs/plots/mse_distribution.png


## 7. Compute and Save Thresholds

In [8]:
import json
normal_errors = compute_reconstruction_errors(autoencoder, X_val_normal, device=DEVICE)
thr_2std = compute_threshold(normal_errors, 2.0)
thr_3std = compute_threshold(normal_errors, 3.0)

print(f"Normal MSE  mean={normal_errors.mean():.5f}  std={normal_errors.std():.5f}")
print(f"Threshold (mean + 2*std) = {thr_2std:.5f}")
print(f"Threshold (mean + 3*std) = {thr_3std:.5f}")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULTS_DIR / "thresholds.json", "w") as f:
    json.dump({"mean_2std": thr_2std, "mean_3std": thr_3std}, f, indent=2)
print(f"Thresholds saved.")

print("\n[DONE] Encoder saved: encoder_pretrained.pth  Decoder: decoder_pretrained.pth")

Normal MSE  mean=0.05250  std=0.01163
Threshold (mean + 2*std) = 0.07575
Threshold (mean + 3*std) = 0.08738
Thresholds saved.

[DONE] Encoder saved: encoder_pretrained.pth  Decoder: decoder_pretrained.pth
